# A programmable two-qubit Clifford+T circuit runner on the surface code

The previous notebook prepared one specific non-Clifford state. Here we build a **general
runner**: you write a two-qubit *logical* circuit as a list of gates and it is executed on
the surface code, with every non-Clifford `T` (and `S`) realised by **gate teleportation**
from a perfect magic state, **interleaved error correction** between gates, and logical
readout validated against an exact classical 2-qubit reference.

- **Initial states** (per patch): `|0⟩, |1⟩, |+⟩, |−⟩`.
- **Gates**: `X, Z, CNOT` (transversal), and `S, T` (**gate teleportation**).
- Each circuit is just a list, e.g. the magic Bell state is
  `run_circuit([(:T,1),(:CNOT,1,2)]; inits=(:plus,:zero))`.

We assume **magic-state distillation is perfect**: a fresh ideal `|A⟩_L = T|+⟩` or
`|Y⟩_L = S|+⟩` is available on demand from a third, recyclable ancilla patch. Guiding
principle: **`T` is the only genuinely non-Clifford operation, so it is the only thing that
must physically inject "magic" into the MPS**; every other gate is Clifford and every
correction is applied physically, so the encoded state always equals the ideal logical state.

*(A leading Hadamard is provided by initialising in `|+⟩`/`|−⟩`; mid-circuit `H` is the one
open piece — see "Logical H" below.)*

## 0 · Setup (surface-code machinery, 3 patches: 2 data + 1 magic ancilla)

Reused from the magic-state notebook with the geometry extended to three patches. `op("T")`/`op("S")` are *physical* single-qubit phase gates (the logical versions are the hard part, handled by teleportation below). Skim if familiar.

In [ ]:
using ITensors, ITensorMPS, LinearAlgebra, Plots, Random
Random.seed!(0xB011)
const threshold = 1e-6     # MPS truncation cutoff
gr();

# Physical T gate for S=1/2 sites (Up=|0>, Dn=|1>): diag(1, e^{iπ/4}).
# (This is a PHYSICAL single-qubit T; the hard part is the LOGICAL T — see §1.)
ITensors.op(::OpName"T", ::SiteType"S=1/2") = ComplexF64[1 0; 0 exp(im*π/4)]

# d = 3 keeps the transversal CNOT (which spans ~half the MPS in our
# side-by-side layout) tractable. Crank up to d = 5 once you've validated
# the rest of the pipeline and are willing to budget the runtime.
const d = 3

# Geometry for a single patch (same as the no-reset notebook)
data_coords = vec([(x, y) for x in 0:(d-1), y in 0:(d-1)])
z_aux_local = Tuple{Float64,Float64}[]
x_aux_local = Tuple{Float64,Float64}[]
for x in 0:(d-2), y in 0:(d-2)
    push!((x + y) % 2 == 0 ? z_aux_local : x_aux_local, (x + 0.5, y + 0.5))
end
for y in 1:2:(d-2);  push!(z_aux_local, (-0.5,    y + 0.5)); end
for y in 0:2:(d-2);  push!(z_aux_local, (d - 0.5, y + 0.5)); end
for x in 0:2:(d-2);  push!(x_aux_local, (x + 0.5, -0.5));    end
for x in 1:2:(d-2);  push!(x_aux_local, (x + 0.5, d - 0.5)); end

const Nz_per_patch = length(z_aux_local)  # 12 for d=5

# MPS ordering: side-by-side. All of patch 1 first, then all of patch 2.
# Within each patch: data, then Z-aux, then X-aux.
# Trade-off: keeps the within-patch CNOT chains short, but the transversal
# CNOT between the two patches now spans roughly half the chain — exactly
# the kind of non-local 2-qubit gate we want to stress-test.
all_keys = Tuple{Int, Tuple{Float64,Float64}}[]
for p in 1:3
    for q in data_coords;  push!(all_keys, (p, q)); end
    for a in z_aux_local;  push!(all_keys, (p, a)); end
    for a in x_aux_local;  push!(all_keys, (p, a)); end
end

sites = siteinds("S=1/2", length(all_keys))
site_of = Dict{Tuple{Int, Tuple{Float64,Float64}}, Index}(
    k => sites[i] for (i, k) in enumerate(all_keys))

# Bookkeeping for diagnostics: index of each site in the MPS
mps_index_of = Dict(k => i for (i, k) in enumerate(all_keys))

println("total sites: ", length(all_keys),
        "  (",  2 * length(data_coords), " data + ",
        2 * length(z_aux_local), " Z-aux + ",
        2 * length(x_aux_local), " X-aux)")
println("transversal CNOT separation: data(p=1,(0,0)) is site ",
        mps_index_of[(1, (0, 0))],
        ", data(p=2,(0,0)) is site ",
        mps_index_of[(2, (0, 0))],
        "  — gap ", mps_index_of[(2, (0, 0))] - mps_index_of[(1, (0, 0))])

# Single-patch data-neighbor lookup
data_neighbors_of(aux_coord) =
    [d for d in data_coords if abs(d[1]-aux_coord[1])==0.5 && abs(d[2]-aux_coord[2])==0.5]

P_up(s) = 0.5 * op("Id", s) + op("Sz", s)
P_dn(s) = 0.5 * op("Id", s) - op("Sz", s)

# Z-basis projective measurement: bit=0 ⇔ Up, bit=1 ⇔ Dn
function measure_Z!(psi, site; cutoff = threshold)
    sz = real(inner(psi', apply(op("Sz", site), psi; cutoff)))
    if rand() < 0.5 + sz
        psi = apply(P_up(site), psi; cutoff); bit = 0
    else
        psi = apply(P_dn(site), psi; cutoff); bit = 1
    end
    bit, psi / sqrt(real(inner(psi, psi)))
end

# One Z-stab measurement on patch p at the given aux coord (no-reset model)
function measure_Z_stab(psi, p::Int, aux_coord; cutoff = threshold)
    aux_site = site_of[(p, aux_coord)]
    nbrs = data_neighbors_of(aux_coord)
    order = length(nbrs) == 4 ? [2, 4, 1, 3] : [1, 2]
    for q in nbrs[order]
        psi = apply(op("CNOT", site_of[(p, q)], aux_site), psi; cutoff)
    end
    measure_Z!(psi, aux_site; cutoff)
end;

# Reset an ancilla to Up (= |0>). Used as a one-shot housekeeping step at
# the start, and between stabilizer-type changes.
function reset_aux!(psi, aux_site; cutoff = threshold)
    sz = real(inner(psi', apply(op("Sz", aux_site), psi; cutoff)))
    if sz < 0
        psi = apply(op("X", aux_site), psi; cutoff)
    end
    psi
end

# One-shot X-stab measurement: H — CNOTs (aux→data) — H — Z-measure aux.
# Assumes the aux qubit has been reset to |0> already.
function measure_X_stab(psi, p::Int, aux_coord; cutoff = threshold)
    aux_site = site_of[(p, aux_coord)]
    psi = apply(op("H", aux_site), psi; cutoff)
    nbrs = data_neighbors_of(aux_coord)
    order = length(nbrs) == 4 ? [2, 1, 4, 3] : [1, 2]
    for q in nbrs[order]
        psi = apply(op("CNOT", aux_site, site_of[(p, q)]), psi; cutoff)
    end
    psi = apply(op("H", aux_site), psi; cutoff)
    measure_Z!(psi, aux_site; cutoff)
end

# Project a patch into its codespace once. Measure every Z-stab and every
# X-stab (each ancilla reset beforehand). Returns the two syndrome vectors.
function project_to_codespace!(psi, p::Int; cutoff = threshold)
    z_syn = Int[]
    for ac in z_aux_local
        psi = reset_aux!(psi, site_of[(p, ac)]; cutoff)
        bit, psi = measure_Z_stab(psi, p, ac; cutoff)
        push!(z_syn, bit)
    end
    x_syn = Int[]
    for ac in x_aux_local
        psi = reset_aux!(psi, site_of[(p, ac)]; cutoff)
        bit, psi = measure_X_stab(psi, p, ac; cutoff)
        push!(x_syn, bit)
    end
    (; z_syn, x_syn), psi
end;

# Run R rounds of Z-stab measurements on both patches, with reset between
# rounds. Optionally inject physical X errors (Dict mapping round -> list of
# (patch, data_coord)) and measurement errors (Dict mapping round ->
# list of (patch, stab_idx)).
#
# Returns:
#   raw[p][r]  = vector of Nz_per_patch bits, the recorded Z-stab outcomes
#                of patch p at round r.
function run_Z_rounds!(psi, R::Int;
        physical_X = Dict{Int, Vector{Tuple{Int, Tuple{Int,Int}}}}(),
        meas_err   = Dict{Int, Vector{Tuple{Int, Int}}}(),
        cutoff     = threshold)
    raw = Dict(1 => [Int[] for _ in 1:R], 2 => [Int[] for _ in 1:R])
    for r in 1:R
        # Physical X errors injected *before* the round-r measurements
        for (p, q) in get(physical_X, r, Tuple{Int, Tuple{Int,Int}}[])
            psi = apply(op("X", site_of[(p, q)]), psi; cutoff)
        end

        for p in 1:2, (i, ac) in enumerate(z_aux_local)
            psi = reset_aux!(psi, site_of[(p, ac)]; cutoff)
            bit, psi = measure_Z_stab(psi, p, ac; cutoff)
            push!(raw[p][r], bit)
        end

        # Apply classical measurement-error flips to the recorded bits
        for (p, i) in get(meas_err, r, Tuple{Int, Int}[])
            raw[p][r][i] = 1 - raw[p][r][i]
        end
    end
    raw, psi
end;

# Apply transversal H to patch p's data qubits.
function logical_H!(psi, p::Int; cutoff = threshold)
    for q in data_coords
        psi = apply(op("H", site_of[(p, q)]), psi; cutoff)
    end
    psi
end

# Apply transversal CNOT(p_ctrl, p_tgt). Each gate spans roughly half the MPS.
function logical_CNOT!(psi, p_ctrl::Int, p_tgt::Int; cutoff = threshold)
    for q in data_coords
        psi = apply(op("CNOT", site_of[(p_ctrl, q)], site_of[(p_tgt, q)]), psi; cutoff)
    end
    psi
end;

# X-stab multi-round readout (detects Z errors), mirror of run_Z_rounds!.
function run_X_rounds!(psi, R::Int;
        physical_Z = Dict{Int, Vector{Tuple{Int, Tuple{Int,Int}}}}(),
        cutoff = threshold)
    raw = Dict(1 => [Int[] for _ in 1:R], 2 => [Int[] for _ in 1:R])
    for r in 1:R
        for (p, q) in get(physical_Z, r, Tuple{Int, Tuple{Int,Int}}[])
            psi = apply(op("Z", site_of[(p, q)]), psi; cutoff)
        end
        for p in 1:2, (i, ac) in enumerate(x_aux_local)
            psi = reset_aux!(psi, site_of[(p, ac)]; cutoff)
            bit, psi = measure_X_stab(psi, p, ac; cutoff)
            push!(raw[p][r], bit)
        end
    end
    raw, psi
end

# Spatial matching-graph bookkeeping for ONE patch (geometry shared by both).
z_aux_containing(q) =
    [i for (i, a) in enumerate(z_aux_local)
       if abs(a[1]-q[1])==0.5 && abs(a[2]-q[2])==0.5]

edge_data_local  = Dict{Tuple{Int,Int}, Tuple{Int,Int}}()
bedge_data_local = Dict{Int, Tuple{Int,Int}}()
for q in data_coords
    zs = z_aux_containing(q)
    if length(zs) == 2
        a, b = minmax(zs[1], zs[2])
        edge_data_local[(a, b)] = q
    elseif length(zs) == 1
        bedge_data_local[zs[1]] = q
    end
end

# 3-D matching graph for a given number of rounds R (reset model -> r↔r+1).
function build_graph(R::Int)
    Nz   = Nz_per_patch
    Ntot = R * Nz + 1
    BND  = Ntot
    node_id(i, r) = (r - 1) * Nz + i
    decode_id(u)  = (mod1(u, Nz), div(u - 1, Nz) + 1)

    dist = fill(Inf, Ntot, Ntot)
    nxt  = fill(-1,  Ntot, Ntot)
    for i in 1:Ntot
        dist[i, i] = 0.0; nxt[i, i] = i
    end
    add!(u, v) = begin
        dist[u, v] = 1.0; dist[v, u] = 1.0
        nxt[u, v]  = v;   nxt[v, u]  = u
    end
    for r in 1:R, ((i, j), _) in edge_data_local
        add!(node_id(i, r), node_id(j, r))
    end
    for r in 1:(R-1), i in 1:Nz
        add!(node_id(i, r), node_id(i, r + 1))
    end
    for r in 1:R, (i, _) in bedge_data_local
        add!(node_id(i, r), BND)
    end
    for k in 1:Ntot, i in 1:Ntot, j in 1:Ntot
        if dist[i, k] + dist[k, j] < dist[i, j]
            dist[i, j] = dist[i, k] + dist[k, j]
            nxt[i, j]  = nxt[i, k]
        end
    end
    function edge_kind(u, v)
        u, v = minmax(u, v)
        if v == BND
            i, _ = decode_id(u); return (:boundary, bedge_data_local[i])
        end
        iu, ru = decode_id(u); iv, rv = decode_id(v)
        if ru == rv
            a, b = minmax(iu, iv); return (:spatial, edge_data_local[(a, b)])
        else
            return (:time, (iu, min(ru, rv)))
        end
    end
    function path(a, b)
        nxt[a, b] == -1 && return Int[]
        p = [a]
        while a != b; a = nxt[a, b]; push!(p, a); end
        p
    end
    (; dist, nxt, BND, node_id, decode_id, edge_kind, path, R)
end;

function mwpm(defects::Vector{Int}, dist::Matrix{Float64}, BND::Int)
    isempty(defects) && return Tuple{Int,Int}[], 0.0
    nd    = length(defects)
    nodes = vcat(defects, fill(BND, nd))
    N     = length(nodes)
    W = zeros(N, N)
    for i in 1:N, j in (i+1):N
        W[i, j] = (nodes[i] == BND && nodes[j] == BND) ? 0.0 :
                  dist[nodes[i], nodes[j]]
        W[j, i] = W[i, j]
    end
    function rec(rem)
        isempty(rem) && return 0.0, Tuple{Int,Int}[]
        first = rem[1]; best_c = Inf; best_m = Tuple{Int,Int}[]
        for k in 2:length(rem)
            c = W[first, rem[k]]
            isfinite(c) || continue
            sub = vcat(rem[2:k-1], rem[k+1:end])
            sc, sm = rec(sub)
            if c + sc < best_c
                best_c = c + sc; best_m = vcat([(first, rem[k])], sm)
            end
        end
        best_c, best_m
    end
    cost, m_idx = rec(collect(1:N))
    pairs = [(nodes[i], nodes[j]) for (i, j) in m_idx
             if !(nodes[i] == BND && nodes[j] == BND)]
    pairs, cost
end

# Decode one patch's syndrome history into a Pauli-frame X-correction list.
# `raw_history[r]` is the round-r vector of bits for this patch.
# `initial_syn` is the round-0 syndrome (from the codespace-projection step),
# included so the first detector compares round 1 against it.
function decode_patch(raw_history::Vector{Vector{Int}}, initial_syn::Vector{Int}, g)
    R = g.R
    # detectors
    prev = initial_syn
    lit  = Int[]
    for r in 1:R
        d = raw_history[r] .⊻ prev
        for (i, b) in enumerate(d)
            b == 1 && push!(lit, g.node_id(i, r))
        end
        prev = raw_history[r]
    end
    pairs, cost = mwpm(lit, g.dist, g.BND)
    x_qubits = Tuple{Int,Int}[]; meas_err = Tuple{Int,Int}[]
    for (a, b) in pairs
        p = g.path(a, b)
        for k in 1:(length(p)-1)
            kind = g.edge_kind(p[k], p[k+1])
            if kind[1] === :time
                push!(meas_err, kind[2])
            else
                push!(x_qubits, kind[2])
            end
        end
    end
    counts = Dict{Tuple{Int,Int}, Int}()
    for q in x_qubits; counts[q] = get(counts, q, 0) + 1; end
    frame = [q for (q, c) in counts if isodd(c)]
    (; lit, pairs, frame, meas_err, cost)
end;

# --- X-stab matching geometry + generalized graph (for Z-error decoding) ---
x_aux_containing(q) =
    [i for (i, a) in enumerate(x_aux_local)
       if abs(a[1]-q[1])==0.5 && abs(a[2]-q[2])==0.5]
const Nx_per_patch = length(x_aux_local)
xedge_data_local  = Dict{Tuple{Int,Int}, Tuple{Int,Int}}()
xbedge_data_local = Dict{Int, Tuple{Int,Int}}()
for q in data_coords
    xs = x_aux_containing(q)
    if length(xs) == 2
        a, b = minmax(xs[1], xs[2]); xedge_data_local[(a, b)] = q
    elseif length(xs) == 1
        xbedge_data_local[xs[1]] = q
    end
end

function build_graph_generic(R::Int, Nstab::Int, edge_data, bedge_data)
    Ntot = R * Nstab + 1; BND = Ntot
    node_id(i, r) = (r - 1) * Nstab + i
    decode_id(u)  = (mod1(u, Nstab), div(u - 1, Nstab) + 1)
    dist = fill(Inf, Ntot, Ntot); nxt = fill(-1, Ntot, Ntot)
    for i in 1:Ntot; dist[i, i] = 0.0; nxt[i, i] = i; end
    add!(u, v) = (dist[u,v]=1.0; dist[v,u]=1.0; nxt[u,v]=v; nxt[v,u]=u)
    for r in 1:R, ((i, j), _) in edge_data; add!(node_id(i, r), node_id(j, r)); end
    for r in 1:(R-1), i in 1:Nstab; add!(node_id(i, r), node_id(i, r + 1)); end
    for r in 1:R, (i, _) in bedge_data; add!(node_id(i, r), BND); end
    for k in 1:Ntot, i in 1:Ntot, j in 1:Ntot
        if dist[i, k] + dist[k, j] < dist[i, j]
            dist[i, j] = dist[i, k] + dist[k, j]; nxt[i, j] = nxt[i, k]
        end
    end
    function edge_kind(u, v)
        u, v = minmax(u, v)
        if v == BND; i, _ = decode_id(u); return (:boundary, bedge_data[i]); end
        iu, ru = decode_id(u); iv, rv = decode_id(v)
        if ru == rv; a, b = minmax(iu, iv); return (:spatial, edge_data[(a, b)]); end
        return (:time, (iu, min(ru, rv)))
    end
    function path(a, b)
        nxt[a, b] == -1 && return Int[]
        p = [a]; while a != b; a = nxt[a, b]; push!(p, a); end; p
    end
    (; dist, nxt, BND, node_id, decode_id, edge_kind, path, R)
end
build_graph_X(R) = build_graph_generic(R, Nx_per_patch, xedge_data_local, xbedge_data_local)

const ZL_support = [(x, 0) for x in 0:(d-1)]

function measure_data_Z(psi_in; cutoff = threshold)
    psi = copy(psi_in)
    bits = Dict{Tuple{Int, Tuple{Int,Int}}, Int}()
    for p in 1:2, q in data_coords
        site = site_of[(p, q)]
        sz = real(inner(psi', apply(op("Sz", site), psi; cutoff)))
        if rand() < 0.5 + sz
            psi = apply(P_up(site), psi; cutoff); bits[(p, q)] = 0
        else
            psi = apply(P_dn(site), psi; cutoff); bits[(p, q)] = 1
        end
        psi = psi / sqrt(real(inner(psi, psi)))
    end
    bits
end

function logical_Z(bits, frame::Vector{Tuple{Int,Int}}, p::Int)
    raw          = mod(sum(bits[(p, q)]            for q in ZL_support), 2)
    frame_parity = mod(count(q -> q in ZL_support, frame), 2)
    raw ⊻ frame_parity
end;

# X-basis readout (rotate with H, then Z-measure) + logical X.
const XL_col0 = [(0, y) for y in 0:(d-1)]   # logical-X support (column x=0)
function measure_data_X(psi_in; cutoff = threshold)
    psi = copy(psi_in)
    for p in 1:2, q in data_coords
        psi = apply(op("H", site_of[(p, q)]), psi; cutoff)
    end
    measure_data_Z(psi; cutoff)     # Z-measure after H = X-basis outcomes
end
function logical_X(bits, zframe::Vector{Tuple{Int,Int}}, p::Int; support = XL_col0)
    raw = mod(sum(bits[(p, q)] for q in support), 2)
    fp  = mod(count(q -> q in support, zframe), 2)
    raw ⊻ fp
end

## Gate teleportation for S and T

`T` is not transversal, so it enters as a state. To apply `T` to a logical qubit on patch
`p`: inject a fresh `|A⟩_L` on the ancilla patch, transversal-CNOT `p → ancilla`, measure
the ancilla in logical `Z` (outcome `m`); if `m=1` a byproduct `S` correction is needed.
Because `S` is itself non-transversal, we apply **it** by the same trick — an `S`-gadget
consuming a `|Y⟩_L` ancilla, whose byproduct is a plain Pauli `Z` (transversal). So the
recursion bottoms out at a physical Pauli and no non-transversal gate is ever applied
directly. In-circuit `S` gates use the same `S`-gadget. This is the **general
gate-teleportation scheme** — the pattern extends to any gate in the Clifford hierarchy
(a level-`k` gate teleports with a level-`(k−1)` byproduct). All ancilla states are prepared
by corner injection (generalised to any seed in `{|0⟩,|1⟩,|+⟩,|−⟩,|A⟩,|Y⟩}`).

In [ ]:
# Teleportation gadgets + injection of arbitrary seed states (on top of base_3patch.jl).
using ITensors
ITensors.op(::OpName"S", ::SiteType"S=1/2") = ComplexF64[1 0; 0 im]

# reset all data qubits of patch p to |0>
function reset_patch_data!(psi, p; cutoff = threshold)
    for q in data_coords
        s = site_of[(p, q)]
        sz = real(inner(psi', apply(op("Sz", s), psi; cutoff)))
        b = rand() < 0.5 + sz ? 0 : 1
        psi = apply(b == 0 ? P_up(s) : P_dn(s), psi; cutoff)
        psi = psi / sqrt(real(inner(psi, psi)))
        b == 1 && (psi = apply(op("X", s), psi; cutoff))
    end
    psi
end

# corner-seed injection of a single-qubit state into patch p.
seed_ops(sym) = sym === :zero ? String[] : sym === :one ? ["X"] :
                sym === :plus ? ["H"]   : sym === :minus ? ["H","Z"] :
                sym === :A ? ["H","T"] : sym === :Y ? ["H","S"] : error("seed $sym")
function inject_seed!(psi, p, sym::Symbol; cutoff = threshold)
    psi = reset_patch_data!(psi, p; cutoff)
    c = site_of[(p, (0, 0))]
    for g in seed_ops(sym); psi = apply(op(g, c), psi; cutoff); end   # seed corner
    for q in XL_col0[2:end]                                           # rest of column 0 -> |+>
        psi = apply(op("H", site_of[(p, q)]), psi; cutoff)
    end
    _, psi = project_to_codespace!(psi, p; cutoff)
    psi
end

# destructive logical-Z measurement of patch p (consumes it); returns (bit, psi)
function measure_patch_Z!(psi, p; cutoff = threshold)
    par = 0
    for q in data_coords
        s = site_of[(p, q)]
        sz = real(inner(psi', apply(op("Sz", s), psi; cutoff)))
        b = rand() < 0.5 + sz ? 0 : 1
        psi = apply(b == 0 ? P_up(s) : P_dn(s), psi; cutoff)
        psi = psi / sqrt(real(inner(psi, psi)))
        q in ZL_support && (par ⊻= b)
    end
    par, psi
end

# logical Paulis on the logical support
apply_ZL!(psi, p; cutoff = threshold) = (for q in ZL_support; psi = apply(op("Z", site_of[(p, q)]), psi; cutoff); end; psi)
apply_XL!(psi, p; cutoff = threshold) = (for q in XL_col0;   psi = apply(op("X", site_of[(p, q)]), psi; cutoff); end; psi)

# S-gate teleportation: |Y> ancilla on patch 3, CNOT(p->3), measure, byproduct Z if m=1.
function teleport_S!(psi, p; cutoff = threshold)
    psi = inject_seed!(psi, 3, :Y; cutoff)
    psi = logical_CNOT!(psi, p, 3; cutoff)
    m, psi = measure_patch_Z!(psi, 3; cutoff)
    m == 1 && (psi = apply_ZL!(psi, p; cutoff))
    psi
end

# T-gate teleportation: |A> ancilla, CNOT(p->3), measure, byproduct S (via S-gadget) if m=1.
# This is the general gate-teleportation scheme; the S byproduct itself is teleported,
# bottoming out at a physical Pauli — so no non-transversal gate is ever applied directly.
function teleport_T!(psi, p; cutoff = threshold)
    psi = inject_seed!(psi, 3, :A; cutoff)
    psi = logical_CNOT!(psi, p, 3; cutoff)
    m, psi = measure_patch_Z!(psi, 3; cutoff)
    m == 1 && (psi = teleport_S!(psi, p; cutoff))
    psi
end

## Logical H (leading vs mid-circuit)

A **leading** Hadamard is free: initialise the qubit in `|+⟩` (or `|−⟩`) instead of `|0⟩`.
That covers the usual "create a superposition, then compute" pattern, and every demo below
uses it.

**Mid-circuit** logical `H` is the one piece left open. Transversal `H` (an `H` on every
data qubit) turns the rotated code into its *dual* — it maps `Z`-plaquettes to `X`-type
operators — so on its own it is not logical `H`; it must be followed by the code's **duality
automorphism** (a lattice symmetry that maps `Z`-plaquette supports back onto `X`-plaquette
supports). For this specific `d=3` boundary layout the natural 90° rotation maps the
plaquettes correctly and gives `H|0⟩_L → |+⟩_L`, but it fails `H·H = I` and is
representative-dependent — i.e. it is *not* the exact automorphism. Getting mid-circuit `H`
right needs a careful boundary-automorphism analysis (or an alternative such as
measurement-based / lattice-surgery `H`); until then the runner raises an error on a
mid-circuit `(:H, p)` rather than return a wrong result.

## The runner + interleaved error correction

`run_circuit(circuit; inits, readout, R, errors)` prepares the two data patches, then for
each gate: applies it (transversally, or by teleportation), **re-baselines** the syndrome
reference (a logical gate legitimately changes the syndrome — CNOT couples patches — so those
changes must not be read as errors), optionally injects a test error, and runs `R` rounds of
**both** `Z`- and `X`-stabilizers on the data patches. `X`-errors are decoded on the `Z`-stab
graph (into an `X`-frame) and `Z`-errors on the `X`-stab graph (into a `Z`-frame). Both
stabilizer types can be extracted every round because they commute (CSS) and commute with the
logical operators, so syndrome extraction never disturbs the logical state. At readout the
accumulated frame is XORed into the logical parity (`Z`-readout uses the `X`-frame;
`X`-readout uses the `Z`-frame).

In [ ]:
# General two-qubit Clifford+T circuit runner with interleaved error correction.
# Depends on: base_3patch.jl, sigma_test.jl (logical_H_ft!), magic_gadgets.jl.

# dispatch a single logical op onto the MPS
function apply_logical!(psi, op; cutoff = threshold)
    g = op[1]
    if     g === :X;    return apply_XL!(psi, op[2]; cutoff)
    elseif g === :Z;    return apply_ZL!(psi, op[2]; cutoff)
    elseif g === :H;    error("mid-circuit logical H is not yet verified for this layout " *
                              "(the rotated-code duality automorphism is unresolved — see the " *
                              "'Logical H' notes). Use a |+> / |-> initial state for a leading H.")
    elseif g === :S;    return teleport_S!(psi, op[2]; cutoff)
    elseif g === :T;    return teleport_T!(psi, op[2]; cutoff)
    elseif g === :CNOT; return logical_CNOT!(psi, op[2], op[3]; cutoff)
    else; error("unknown gate $g"); end
end

# one round of syndrome extraction (both stabilizer types) on data patches 1,2
function extract_both(psi; cutoff = threshold)
    sZ = Dict{Int,Vector{Int}}(); sX = Dict{Int,Vector{Int}}()
    for p in 1:2
        z = Int[]
        for ac in z_aux_local
            psi = reset_aux!(psi, site_of[(p, ac)]; cutoff)
            b, psi = measure_Z_stab(psi, p, ac; cutoff); push!(z, b)
        end
        x = Int[]
        for ac in x_aux_local
            psi = reset_aux!(psi, site_of[(p, ac)]; cutoff)
            b, psi = measure_X_stab(psi, p, ac; cutoff); push!(x, b)
        end
        sZ[p] = z; sX[p] = x
    end
    sZ, sX, psi
end

# Run a logical circuit with interleaved EC. Returns (l1, l2) logical bits in `readout` basis.
#   circuit :: Vector of ops (:H,p)/(:S,p)/(:T,p)/(:X,p)/(:Z,p)/(:CNOT,c,t)
#   inits   :: (Symbol, Symbol) initial states of patches 1,2
#   readout :: :Z or :X (measured on both patches)
#   errors  :: Dict(gate_index => [(patch, coord, :X|:Z)]) injected right after that gate
#   R       :: EC rounds after each gate (single-round spatial decode; R≥1)
function run_circuit(circuit; inits = (:zero, :zero), readout = :Z, R = 1,
                     errors = Dict{Int, Vector{Tuple{Int, Tuple{Int,Int}, Symbol}}}(),
                     cutoff = threshold)
    psi = MPS(sites, "Up")
    psi = inject_seed!(psi, 1, inits[1]; cutoff)
    psi = inject_seed!(psi, 2, inits[2]; cutoff)
    xframe = Dict(1 => Tuple{Int,Int}[], 2 => Tuple{Int,Int}[])   # X-error corrections
    zframe = Dict(1 => Tuple{Int,Int}[], 2 => Tuple{Int,Int}[])   # Z-error corrections
    gZ = build_graph(1); gX = build_graph_X(1)

    for (gi, g) in enumerate(circuit)
        psi = apply_logical!(psi, g; cutoff)
        # Re-baseline the syndrome reference AFTER the gate: a logical gate legitimately
        # changes the syndrome (σ permutes stabilizer labels; CNOT couples patches), so we
        # must not read those changes as errors. The following R rounds then catch any error
        # that appears after this baseline.
        refZ, refX, psi = extract_both(psi; cutoff)
        for (p, q, kind) in get(errors, gi, Tuple{Int, Tuple{Int,Int}, Symbol}[])
            psi = apply(op(kind === :X ? "X" : "Z", site_of[(p, q)]), psi; cutoff)
        end
        for _ in 1:R
            sZ, sX, psi = extract_both(psi; cutoff)
            for p in 1:2
                append!(xframe[p], decode_patch([sZ[p]], refZ[p], gZ).frame)
                append!(zframe[p], decode_patch([sX[p]], refX[p], gX).frame)
            end
            refZ, refX = sZ, sX
        end
    end

    # readout in the requested basis (all patches in standard frame → logicals commute with stabs)
    bits = readout === :X ? measure_data_X(psi; cutoff) : measure_data_Z(psi; cutoff)
    if readout === :X
        l1 = logical_X(bits, zframe[1], 1); l2 = logical_X(bits, zframe[2], 2)
    else
        l1 = logical_Z(bits, xframe[1], 1); l2 = logical_Z(bits, xframe[2], 2)
    end
    (; l1, l2)
end

## Validation against an exact classical simulator

`ref_state`/`ref_corr` are a tiny exact 4-dimensional two-qubit statevector simulator — the
ground truth. For each circuit we compare the surface-code logical correlations `⟨O₁O₂⟩`
in the `Z` and `X` bases against the exact values; they agree within sampling error across
Clifford-only, single-`T`, two-`T`, and mixed Clifford+`T` circuits, and the interleaved
decoder catches injected mid-circuit errors.

In [ ]:
# Exact classical 2-qubit statevector reference — ground truth for the surface-code runner.
# A circuit is a list of ops: (:H,p) (:S,p) (:T,p) (:X,p) (:Z,p) (:CNOT,c,t), p,c,t ∈ {1,2}.
# inits :: NTuple{2,Symbol} in {:0,:1,:+,:-,:A,:Y}.
using LinearAlgebra

const _ket = Dict(
    :zero => ComplexF64[1, 0], :one => ComplexF64[0, 1],
    :plus => ComplexF64[1, 1]/√2, :minus => ComplexF64[1, -1]/√2,
    :A => ComplexF64[1, exp(im*π/4)]/√2, :Y => ComplexF64[1, im]/√2)
const _H = ComplexF64[1 1; 1 -1]/√2
const _S = ComplexF64[1 0; 0 im]
const _T = ComplexF64[1 0; 0 exp(im*π/4)]
const _X = ComplexF64[0 1; 1 0]
const _Z = ComplexF64[1 0; 0 -1]
const _I = ComplexF64[1 0; 0 1]
_1q(g, p) = p == 1 ? kron(g, _I) : kron(_I, g)
const _CNOT12 = ComplexF64[1 0 0 0; 0 1 0 0; 0 0 0 1; 0 0 1 0]  # ctrl 1, tgt 2
const _CNOT21 = ComplexF64[1 0 0 0; 0 0 0 1; 0 0 1 0; 0 1 0 0]  # ctrl 2, tgt 1

function ref_state(circuit; inits = (:zero, :zero))
    ψ = kron(_ket[inits[1]], _ket[inits[2]])
    for op in circuit
        g = op[1]
        if g == :H;    ψ = _1q(_H, op[2]) * ψ
        elseif g == :S; ψ = _1q(_S, op[2]) * ψ
        elseif g == :T; ψ = _1q(_T, op[2]) * ψ
        elseif g == :X; ψ = _1q(_X, op[2]) * ψ
        elseif g == :Z; ψ = _1q(_Z, op[2]) * ψ
        elseif g == :CNOT; ψ = (op[2] == 1 ? _CNOT12 : _CNOT21) * ψ
        else; error("unknown op $g"); end
    end
    ψ
end

# <P1 ⊗ P2> for single-qubit Paulis given by symbol (:X,:Y,:Z,:I)
const _P = Dict(:X => _X, :Y => ComplexF64[0 -im; im 0], :Z => _Z, :I => _I)
ref_corr(ψ, b1::Symbol, b2::Symbol) = real(ψ' * kron(_P[b1], _P[b2]) * ψ)

## Demo · run circuits and validate against the exact reference

Each circuit is a gate list; we compare surface-code logical correlations to the exact 2-qubit reference in the Z and X bases, then show interleaved EC catching an injected error. (Small N, d=3, 3 patches — this is the slow cell.)

In [ ]:

using Statistics, Random; Random.seed!(2024)
ev(l) = 1 - 2*l
sc(c, ini, b; n=20, errors=Dict()) =
    mean(begin o = run_circuit(c; inits=ini, readout=b, errors=errors); ev(o.l1)*ev(o.l2) end for _ in 1:n)
N = 20
demos = [
  ("Clifford Bell  (|+>,|0>)·CNOT", [(:CNOT,1,2)],                       (:plus,:zero)),
  ("magic Bell     T·CNOT",         [(:T,1),(:CNOT,1,2)],                (:plus,:zero)),
  ("two-T (T²=S)    T·T·CNOT",       [(:T,1),(:T,1),(:CNOT,1,2)],         (:plus,:zero)),
  ("mixed          S·T·CNOT·T",     [(:S,1),(:T,1),(:CNOT,1,2),(:T,2)],  (:plus,:zero)),
]
println("\n### circuit runner vs exact classical reference  (⟨O₁O₂⟩) ###")
for (nm, c, ini) in demos
    ψ = ref_state(c; inits = ini)
    z = sc(c, ini, :Z; n=N); x = sc(c, ini, :X; n=N)
    println(rpad(nm,32), " Z sim=", lpad(round(z,digits=2),5), " ref=", lpad(round(ref_corr(ψ,:Z,:Z),digits=2),5),
            "  |  X sim=", lpad(round(x,digits=2),5), " ref=", lpad(round(ref_corr(ψ,:X,:X),digits=2),5))
end
println("\n### interleaved EC catches a mid-circuit error (magic Bell) ###")
mb = [(:T,1),(:CNOT,1,2)]; ini = (:plus,:zero); ψ = ref_state(mb; inits=ini)
zc = sc(mb, ini, :Z; n=N, errors = Dict(2 => [(2,(1,1),:X)]))
xc = sc(mb, ini, :X; n=N, errors = Dict(2 => [(2,(1,1),:Z)]))
println("  X error after CNOT → Z-readout ⟨Z₁Z₂⟩ = ", round(zc,digits=2), " (ref ", round(ref_corr(ψ,:Z,:Z),digits=2), ")")
println("  Z error after CNOT → X-readout ⟨X₁X₂⟩ = ", round(xc,digits=2), " (ref ", round(ref_corr(ψ,:X,:X),digits=2), ")")
println("\nDONE")

## Scope, limitations, next steps

- **What works (validated).** Any circuit of the form *single-qubit-basis initialisation →
  a sequence of `{X, Z, S, T, CNOT}`* runs correctly, with `T`/`S` teleported from perfect
  magic states and interleaved error correction throughout. Adding more logical qubits is
  "just" more patches (the ancilla is already a third patch).
- **The recurring theme.** Syndrome extraction, decoding and byproduct bookkeeping are all
  Clifford; the MPS carries the genuine non-stabilizer content injected by the `T` gates and
  is what eventually limits classical simulation.
- **Open / simplified.** *Mid-circuit `H`* (the duality automorphism, above). *Single-round
  spatial decoding* between gates (catches data errors, not measurement errors — use `R≥2`
  with the time-edge graph for those). *Idealised ancilla* (perfect magic, measured cleanly).
  Readout supports `:Z`/`:X` (`:Y` = a logical `S†` before an `X`-readout). `d=3` and small
  shot counts keep the 3-patch, 51-site runs tractable — the demo cell is the slow one.
- **Next.** Resolve mid-circuit `H`; distillation instead of ideal injection;
  measurement-error-aware multi-round decoding between gates; the stochastic noise-model /
  threshold sweep ("item 2").